<a href="https://colab.research.google.com/github/Quantium/Zeus-Notebooks/blob/main/ZeusPipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Installations
# !pip install rich #Already in colab vms
!pip install rich-tools pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: rich
    Found existing installation: rich 13.9.4
    Uninstalling rich-13.9.4:
      Successfully uninstalled rich-13.9.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.


In [ ]:
#@title Base Libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [ ]:
#@title Select for Dark Mode
import seaborn as sns

darkmode_selector = True #@param {type:"boolean"}
if darkmode_selector:
  plt.style.use('seaborn-v0_8-whitegrid')

  plt.style.use('dark_background')
  plt.rcParams.update({
      'axes.edgecolor': 'white',
      'axes.labelcolor': 'white',
      'text.color': 'white',
      'xtick.color': 'white',
      'ytick.color': 'white',
      'grid.color': 'lightgray',
      'grid.linestyle': '--',
      'grid.linewidth': 0.5,
      'lines.linewidth': 1.5,
      'lines.markersize': 5,
      'font.size': 10,
      'figure.figsize': (8, 6),
      'savefig.dpi': 300,
      'savefig.bbox': 'tight',
      'legend.edgecolor': 'white',
      'legend.frameon': True,
      'legend.fancybox': False,
  })

  #Para as matrices de correlación
  dark_cmap = sns.color_palette("binary", as_cmap=True)

In [ ]:
#@title ML Libraries

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV,RandomizedSearchCV
from sklearn.feature_selection import SelectKBest, chi2, VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score, RocCurveDisplay,accuracy_score, roc_curve, auc, classification_report,confusion_matrix,precision_recall_curve, precision_recall_fscore_support,fbeta_score, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, LinearRegression, HuberRegressor

from scipy.stats import kurtosis, skew

In [ ]:
#@title Función para guardar objetos con joblib
## Crea o carga un objeto, ideal para objetos que toman mucho tiempo en generarse
def create_or_load(folder_path, filename, creation_function):
  file_path=os.path.join(folder_path, filename)
  if not os.path.exists(file_path):
      print(f"Joblib file not found. Creating {file_path}")
      data = creation_function()
      joblib.dump(data, file_path)

  print(f"Joblib file found.{file_path}")
  return joblib.load(file_path)

In [ ]:
#@title Google Drive
from google.colab import drive
from google.colab import auth
import gspread
from google.auth import default
import pandas as pd

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

drive.mount('/content/drive')
working_folder = "/content/drive/MyDrive/Darovi Consulting/Zeus/reextraccindedatosproyectohades" # @param {type:"string"}

working_file = 'Columnas para Recibo OC  lineas y pedido'

Mounted at /content/drive


# Ordenes de Compra Lineas


WITH ARTICULO_DETALLE AS (
SELECT A.CODIGO_ARTICULO,
FF.DESCRIPCION FORMA_FARMACEUTICA,
F1.DESCRIPCION D_CODIGO_FAMILIA,
F3.DESCRIPCION SUSTANCIA_ACTIVA,
F4.DESCRIPCION CONTROLADO,
F6.DESCRIPCION TEMPERATURA,
F10.DESCRIPCION CLASIFICACION,
F10.DESCRIPCION SUB_FAMILIA
FROM ARTICULOS A
LEFT JOIN fe_forma_farmaceutica_sat FF
ON FF.CLAVE=A.AUXILIAR_1
LEFT JOIN FAMILIAS F1
ON F1.CODIGO_FAMILIA=A.CODIGO_FAMILIA
AND F1.NUMERO_TABLA=1
AND F1.ULTIMO_NIVEL='S'
AND F1.CODIGO_EMPRESA=A.CODIGO_EMPRESA
LEFT JOIN FAMILIAS F3
ON F3.CODIGO_FAMILIA=A.CODIGO_ESTAD3
AND F3.NUMERO_TABLA=3
AND F3.ULTIMO_NIVEL='S'
AND F3.CODIGO_EMPRESA=A.CODIGO_EMPRESA
LEFT JOIN FAMILIAS F4
ON F4.CODIGO_FAMILIA=A.CODIGO_ESTAD4
AND F4.NUMERO_TABLA=4
AND F4.ULTIMO_NIVEL='S'
AND F4.CODIGO_EMPRESA=A.CODIGO_EMPRESA
LEFT JOIN FAMILIAS F6
ON F6.CODIGO_FAMILIA=A.CODIGO_ESTAD6
AND F6.NUMERO_TABLA=6
AND F6.ULTIMO_NIVEL='S'
AND F6.CODIGO_EMPRESA=A.CODIGO_EMPRESA
WHERE A.CODIGO_EMPRESA = '01'
LEFT JOIN FAMILIAS F11
ON F10.CODIGO_FAMILIA=A.CODIGO_ESTAD11
AND F10.NUMERO_TABLA=11
AND F10.ULTIMO_NIVEL='S'
AND F10.CODIGO_EMPRESA=A.CODIGO_EMPRESA
WHERE A.CODIGO_EMPRESA = '01'
LEFT JOIN FAMILIAS F12
ON F10.CODIGO_FAMILIA=A.CODIGO_ESTAD12
AND F10.NUMERO_TABLA=12
AND F10.ULTIMO_NIVEL='S'
AND F10.CODIGO_EMPRESA=A.CODIGO_EMPRESA
WHERE A.CODIGO_EMPRESA = '01'
)
SELECT PCL.*,
AD.*
FROM PEDIDOS_COMPRAS PC,
PEDIDOS_COMPRAS_LIN PCL,
ARTICULO_DETALLE AD
WHERE PCL.CODIGO_EMPRESA=PC.CODIGO_EMPRESA
AND PCL.ORGANIZACION_COMPRAS=PC.ORGANIZACION_COMPRAS
AND PCL.SERIE_NUMERACION=PC.SERIE_NUMERACION
AND PCL.NUMERO_PEDIDO=PC.NUMERO_PEDIDO
AND PCL.CODIGO_ARTICULO=AD.CODIGO_ARTICULO
AND PC.CODIGO_EMPRESA = '01'
--AND PC.FECHA_PEDIDO >= '05/05/2026'


In [ ]:
#@title Líneas DataFrame
lineas = gc.open(working_file).worksheet('OC lineas') #PEDIDOS_COMPRAS_LIN

rows = lineas.get_all_values()
lineasSucios_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
lineas_df=lineasSucios_df[['DATE','ID','CODIGO_EMPRESA','ORGANIZACION_COMPRAS','SERIE_NUMERACION','NUMERO_PEDIDO','CODIGO_ARTICULO','DESCRIPCION','FECHA_ENTREGA','FECHA_ENTREGA_CONFIRM','UNIDADES_PEDIDAS','STATUS_CIERRE','STATUS_APROBADO','FORMA_FARMACEUTICA','D_CODIGO_FAMILIA']]
lineas_df

,DATE,ID,CODIGO_EMPRESA,ORGANIZACION_COMPRAS,SERIE_NUMERACION,NUMERO_PEDIDO,CODIGO_ARTICULO,DESCRIPCION,FECHA_ENTREGA,FECHA_ENTREGA_CONFIRM,UNIDADES_PEDIDAS,STATUS_CIERRE,STATUS_APROBADO,FORMA_FARMACEUTICA,D_CODIGO_FAMILIA
0,,,1,5450,PCM,29901,010.000.0566.00,METILDOPA ( L-ALFAMETILDOPA) TABLETAS 250 MG E...,05/05/2026,,460,E,S,,MEDICAMENTOS
1,,,1,5450,PCM,29863,010.000.0569.00,NITROPRUSIATO DE SODIO SOLUCIÓN INYEC. 50 MG E...,04/05/2026,,5,E,S,,MEDICAMENTOS
2,,,1,5450,PCM,29864,010.000.0569.00,NITROPRUSIATO DE SODIO SOLUCIÓN INYEC. 50 MG E...,04/05/2026,,3,E,S,,MEDICAMENTOS
3,,,1,5450,PCM,29870,010.000.0569.00,NITROPRUSIATO DE SODIO SOLUCIÓN INYEC. 50 MG E...,04/05/2026,,3,E,S,,MEDICAMENTOS
4,,,1,5450,PCM,29829,010.000.2308.00,FUROSEMIDA. SOLUCIÓN INYECTABLE. CADA AMPOLLET...,04/05/2026,,5018,E,S,,MEDICAMENTOS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8283,,,1,5450,PCM,29854,010.000.6231.00,PERINDOPRIL/AMLODIPINO. COMPRIMIDOS CADA COMPR...,04/05/2026,,10,E,S,,MEDICAMENTOS
8284,,,1,5450,PCM,29869,060.182.0160,CEMENTOS. IONÓMERO DE VIDRIO I. PARA CEMENTACI...,04/05/2026,,27,E,S,,MATERIAL DE CURACIÓN
8285,,,1,5460,OP6,1706,25411UG01438,"CATETER VENOSO CENTRAL PICC, DE POLIURETANO, 5...",04/05/2026,,6,Z,S,,MATERIAL DE CURACIÓN
8286,,,1,5460,OP6,1704,25411UG00795,TERMOMETRO DIGITAL PUNTA RIGIDA SUAVE AL TACTO...,04/05/2026,,1080,Z,S,,MATERIAL DE CURACIÓN


# Ordenes de Compra Cabecera
SELECT PC.*
FROM PEDIDOS_COMPRAS PC
WHERE PC.CODIGO_EMPRESA='01'
--AND PC.FECHA_PEDIDO>=TO_DATE('05/05/2026','DD/MM/YYYY')

In [ ]:
#@title Pedidos DataFrame
pedidos = gc.open(working_file).worksheet(' OC pedidos') #PEDIDOS_COMPRAS

rows = pedidos.get_all_values()
pedidosSucio_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
pedidos_df=pedidosSucio_df[['DATE','ID','CODIGO_EMPRESA','ORGANIZACION_COMPRAS','TIPO_PEDIDO','SERIE_NUMERACION','NUMERO_PEDIDO','CODIGO_PROVEEDOR','FECHA_PEDIDO','CODIGO_ALMACEN','USUARIO_ALTA','FECHA_ALTA','STATUS_CIERRE','STATUS_APROBADO']]
pedidos_df

,DATE,ID,CODIGO_EMPRESA,ORGANIZACION_COMPRAS,TIPO_PEDIDO,SERIE_NUMERACION,NUMERO_PEDIDO,CODIGO_PROVEEDOR,FECHA_PEDIDO,CODIGO_ALMACEN,USUARIO_ALTA,FECHA_ALTA,STATUS_CIERRE,STATUS_APROBADO
0,,,1,7151,M,D51,2093,804210,05/05/2026,ME004,PEVAJE,05/05/2026,E,N
1,,,1,7151,M,D51,2094,804210,05/05/2026,ME004,PEVAJE,05/05/2026,E,N
2,,,1,7151,M,D51,2095,804210,05/05/2026,ME004,PEVAJE,05/05/2026,E,N
3,,,1,7151,M,D51,2096,804210,05/05/2026,ME004,PEVAJE,05/05/2026,E,N
4,,,1,7151,M,D51,2097,804210,05/05/2026,ME004,PEVAJE,05/05/2026,E,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170,,,1,5460,M,OP6,1782,250,06/05/2026,5460D,GOMFLOJED,05/05/2026,E,S
171,,,1,5460,M,OP6,1783,250,06/05/2026,5460D,GOMFLOJED,05/05/2026,E,S
172,,,1,5460,M,OP6,1784,250,06/05/2026,5460D,GOMFLOJED,05/05/2026,E,S
173,,,1,5460,M,OP6,1785,250,06/05/2026,5460D,GOMFLOJED,05/05/2026,E,S


# Calendario de Citas
SELECT
    CEC.NUMERO_EXPEDIENTE,
    CL.FECHA_INICIO,
    CL.LIBREA01 HORA_INICIO,
    PC.D_CODIGO_PROVEEDOR,
    PCL.SERIE_NUMERACION,
PCL.NUMERO_PEDIDO,
PCL.CODIGO_ARTICULO,
PCL.DESCRIPCION,
PCL.PRESENTACION_PEDIDO,
    SUM(PCL.UNIDADES_PEDIDAS) CANTIDAD,
SUM(SUM(PCL.UNIDADES_PEDIDAS)) OVER (
        PARTITION BY CEC.NUMERO_EXPEDIENTE
    ) PIEZAS_TOTALES
FROM CRMEXPEDIENTES_CAB CEC
JOIN (
    SELECT
        PC.*,
        (
            SELECT PRLV.NOMBRE
            FROM PROVEEDORES PRLV
            WHERE PRLV.CODIGO_RAPIDO = PC.CODIGO_PROVEEDOR
              AND PRLV.CODIGO_EMPRESA = PC.CODIGO_EMPRESA
        ) D_CODIGO_PROVEEDOR
    FROM PEDIDOS_COMPRAS PC
) PC
    ON PC.CODIGO_EMPRESA = CEC.EMPRESA
   AND PC.RESERVADO_NUMBER1 = CEC.NUMERO_EXPEDIENTE
JOIN CALENDARIOS_LIBRA CL
    ON CL.LIBREN01 = CEC.NUMERO_EXPEDIENTE
JOIN PEDIDOS_COMPRAS_LIN PCL
    ON PCL.CODIGO_EMPRESA = PC.CODIGO_EMPRESA
   AND PCL.ORGANIZACION_COMPRAS = PC.ORGANIZACION_COMPRAS
   AND PCL.SERIE_NUMERACION = PC.SERIE_NUMERACION
   AND PCL.NUMERO_PEDIDO = PC.NUMERO_PEDIDO
WHERE CEC.TIPO_EXPEDIENTE = '70011'
  --AND CL.CODIGO_CALENDARIO = '2'
AND CL.TIPO_CALENDARIO= 'EMP'
  --AND CL.FECHA_INICIO > TO_DATE('01/04/2026', 'DD/MM/YYYY')
GROUP BY
    CEC.NUMERO_EXPEDIENTE,
    CL.FECHA_INICIO,
    CL.LIBREA01,
    PC.D_CODIGO_PROVEEDOR,
    PCL.SERIE_NUMERACION,
PCL.NUMERO_PEDIDO,
   PCL.CODIGO_ARTICULO,
    PCL.DESCRIPCION,
    PCL.PRESENTACION_PEDIDO
ORDER BY
    1, 3, 4

In [ ]:
#@title Calendario de Citas DataFrame
citas = gc.open(working_file).worksheet('Calendario de Citas')

rows = citas.get_all_values()
citas_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
citas_df

,FECHA_INICIO,HORA_INICIO,D_CODIGO_PROVEEDOR,SERIE_NUMERACION,CODIGO_ARTICULO,DESCRIPCION,PRESENTACION_PEDIDO,NUMERO_EXPEDIENTE,NUMERO_PEDIDO,CANTIDAD,PIEZAS_TOTALES
0,28/09/2027,9:00,COMERCIALIZADORA DE PRODUCTOS INSTITU,PCM,010.000.6344.00,CEFIXIMA.SUSPENSION ORAL CADA 5 ML DE SUSPENSI...,PZ,674166,22083,422,422
1,28/09/2027,10:00,ABBOTT LABORATORIES DE MEXICO SA DE CV,PCM,010.000.6115.00,SELEGILINA TABLETA. CADA TABLETA CONTIENE: CLO...,PZ,674193,22084,1,894
2,28/09/2027,10:00,ABBOTT LABORATORIES DE MEXICO SA DE CV,PCM,010.000.4118.01,ISOSORBIDA DINITRATO DE. SOLUCIÓN INYECTABLE C...,EV,674193,22085,52,894
3,28/09/2027,10:00,ABBOTT LABORATORIES DE MEXICO SA DE CV,PCM,010.000.4188.01,PANCREATINA CAPSULA O GRAGEA CON CAPA ENTERICA...,EV,674193,22086,25,894
4,28/09/2027,10:00,ABBOTT LABORATORIES DE MEXICO SA DE CV,PCM,010.000.6134.01,FENOFIBRATO. CAPSULA CADA CÁPSULA CONTIENE: FE...,EV,674193,22087,112,894
5,28/09/2027,10:00,ABBOTT LABORATORIES DE MEXICO SA DE CV,PCM,010.000.6099.01,LACTULOSA. JARABE. CADA 100 ML CONTIENEN: LACT...,EV,674193,22088,533,894
6,28/09/2027,10:00,ABBOTT LABORATORIES DE MEXICO SA DE CV,PCM,010.000.2620.00,ACIDO VALPROICO CAP. 250MG EV C/60 CAPSULAS,EV,674193,22089,171,894
7,11/06/2026,7:00,"BOEHRINGER INGELHEIM MEXICO, S.A. DE C.V",PCM,010.000.5117.00,TENECTEPLASA SOLUCION INYECTABLE 50.0 MG FRASC...,EV,698624,27668,9,9
8,17/06/2026,7:00,"LABORATORIOS ALPHARMA, S.A DE C.V",PCM,010.000.1206.00,BUTILHIOSCINA O HIOSCINA. GRAGEA O TABLETA CAD...,EV,700530,27846,3452,3452
9,27/02/2027,9:00,"PRODUCTOS FARMACEUTICOS, S.A DE C.V.",PCM,010.000.0865.00,PERMETRINA SOLUCION 1 G. EV CON 110 ML,EV,702340,28175,89,89


SELECT *
FROM RF_TAREAS_01
WHERE FECHA_GEN>=TO_DATE('05/05/2026','DD/MM/YYYY')

In [ ]:
#@title Entradas Logisticas DataFrame
entradas = gc.open(working_file).worksheet('Entradas Logisticas')

rows = entradas.get_all_values()
entradas_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
entradas_df

,EMPRESA,CONTADOR,FECHA_GEN,USUARIO_GEN,MAQUINA_GEN,FECHA_INICIO,FECHA_FIN,MAQUINA,USUARIO,OBSERVACIONES,...,STATUS_ANTERIOR,ASIGNACION_UBICACION,PRIORIDAD_ANTERIOR,TIPO_TAREA_BLOQUEO,TAREA_BLOQUEO,MOTIVO_BLOQUEO_ANT,TIPO_COLOCACION,TRABAJADOR,ZONA_INTERMEDIA,UBICACION_DESTINO_INTERMEDIA
0,1,809,28/11/2025 08:23:57 AM,101684,22,28/11/2025 12:42:35 PM,28/11/2025 12:42:55 PM,0B,JEPEALBR,,...,P,M,50,,,,N,JEPEALBR,,
1,1,810,28/11/2025 08:27:45 AM,HERESA,24,28/11/2025 12:39:32 PM,28/11/2025 12:39:52 PM,1K,HESAMA,,...,P,M,50,,,,N,HESAMA,,
2,1,811,28/11/2025 08:29:12 AM,101684,22,28/11/2025 12:39:59 PM,28/11/2025 12:41:17 PM,1K,HESAMA,,...,P,M,50,,,,N,HESAMA,,
3,1,812,28/11/2025 08:31:05 AM,HERESA,24,28/11/2025 12:41:24 PM,28/11/2025 12:41:46 PM,1K,HESAMA,,...,P,M,50,,,,N,HESAMA,,
4,1,813,28/11/2025 08:32:13 AM,101684,22,28/11/2025 12:38:29 PM,28/11/2025 12:38:51 PM,0B,JEPEALBR,,...,P,M,50,,,,N,JEPEALBR,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6214,1,4785,31/03/2026 12:12:36 PM,HERESA,24,01/04/2026 03:30:59 PM,01/04/2026 03:31:42 PM,0N,ESSALU,,...,P,M,50,,,,N,ESSALU,,
6215,1,4786,31/03/2026 12:14:06 PM,HERESA,24,01/04/2026 03:44:24 PM,01/04/2026 03:44:48 PM,0N,MONUCAM,,...,P,M,50,,,,N,MONUCAM,,
6216,1,5266,10/04/2026 11:10:02 AM,GOCAMA,23,10/04/2026 01:14:34 PM,10/04/2026 01:15:09 PM,IA,FRROCR,,...,P,A,50,,,,N,FRROCR,,
6217,1,5268,10/04/2026 11:10:37 AM,GOCAMA,23,10/04/2026 01:15:18 PM,10/04/2026 01:15:45 PM,IA,FRROCR,,...,P,A,50,,,,N,FRROCR,,


SELECT *
FROM RF_TAREAS_02
WHERE FECHA_GEN>=TO_DATE('05/05/2026','DD/MM/YYYY')

In [ ]:
#@title Ubicaciones DataFrame
Ubicaciones = gc.open(working_file).worksheet('Ubicaciones')

rows = Ubicaciones.get_all_values()
Ubicaciones_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
Ubicaciones_df

,EMPRESA,CONTADOR,FECHA_GEN,USUARIO_GEN,MAQUINA_GEN,FECHA_INICIO,FECHA_FIN,MAQUINA,USUARIO,OBSERVACIONES,...,STATUS_ANTERIOR,ASIGNACION_UBICACION,PRIORIDAD_ANTERIOR,TIPO_TAREA_BLOQUEO,TAREA_BLOQUEO,MOTIVO_BLOQUEO_ANT,TIPO_COLOCACION,TRABAJADOR,ZONA_INTERMEDIA,UBICACION_DESTINO_INTERMEDIA
0,1,809,28/11/2025 08:23:57 AM,101684,22,28/11/2025 12:42:35 PM,28/11/2025 12:42:55 PM,0B,JEPEALBR,,...,P,M,50,,,,N,JEPEALBR,,
1,1,810,28/11/2025 08:27:45 AM,HERESA,24,28/11/2025 12:39:32 PM,28/11/2025 12:39:52 PM,1K,HESAMA,,...,P,M,50,,,,N,HESAMA,,
2,1,811,28/11/2025 08:29:12 AM,101684,22,28/11/2025 12:39:59 PM,28/11/2025 12:41:17 PM,1K,HESAMA,,...,P,M,50,,,,N,HESAMA,,
3,1,812,28/11/2025 08:31:05 AM,HERESA,24,28/11/2025 12:41:24 PM,28/11/2025 12:41:46 PM,1K,HESAMA,,...,P,M,50,,,,N,HESAMA,,
4,1,813,28/11/2025 08:32:13 AM,101684,22,28/11/2025 12:38:29 PM,28/11/2025 12:38:51 PM,0B,JEPEALBR,,...,P,M,50,,,,N,JEPEALBR,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6214,1,4785,31/03/2026 12:12:36 PM,HERESA,24,01/04/2026 03:30:59 PM,01/04/2026 03:31:42 PM,0N,ESSALU,,...,P,M,50,,,,N,ESSALU,,
6215,1,4786,31/03/2026 12:14:06 PM,HERESA,24,01/04/2026 03:44:24 PM,01/04/2026 03:44:48 PM,0N,MONUCAM,,...,P,M,50,,,,N,MONUCAM,,
6216,1,5266,10/04/2026 11:10:02 AM,GOCAMA,23,10/04/2026 01:14:34 PM,10/04/2026 01:15:09 PM,IA,FRROCR,,...,P,A,50,,,,N,FRROCR,,
6217,1,5268,10/04/2026 11:10:37 AM,GOCAMA,23,10/04/2026 01:15:18 PM,10/04/2026 01:15:45 PM,IA,FRROCR,,...,P,A,50,,,,N,FRROCR,,


SELECT *
FROM RF_TAREAS_04
WHERE FECHA_GEN>=TO_DATE('05/05/2026','DD/MM/YYYY')

In [ ]:
#@title Surtido DataFrame
Surtido = gc.open(working_file).worksheet('Surtido')

rows = Surtido.get_all_values()
Surtido_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
Surtido_df

,EMPRESA,CONTADOR,FECHA_GEN,USUARIO_GEN,MAQUINA_GEN,FECHA_INICIO,FECHA_FIN,MAQUINA,USUARIO,OBSERVACIONES,...,MOVIMIENTO_PREPARACION,MOVIMIENTO_ENVIO_BOX,DOCUMENTO_PREPARACION,DOCUMENTO_ENVIO_BOX,ID_AGRUPACION,TRABAJADOR,ASIGNACION_MAQUINA,ROTURA,FECHA_ENVIO_EXT,USUARIO_ENVIO_EXT
0,1,19,#######,NASAAI,AN,#######,#######,PB,ESMEEDU,,...,PREIS,ENVBX,19545367,19545429,$nag$00$npe$4355$spe$P62$epe$2026$org$5460$end$,ESMEEDU,0,,,
1,1,20,#######,NASAAI,AN,#######,#######,PQ,EDESCA,,...,PREIS,ENVBX,19544934,19545871,$nag$00$npe$4355$spe$P62$epe$2026$org$5460$end$,EDESCA,0,,,
2,1,21,#######,NASAAI,AN,#######,#######,PQ,EDESCA,,...,PREIS,ENVBX,19545699,19545807,$nag$00$npe$4355$spe$P62$epe$2026$org$5460$end$,EDESCA,0,,,
3,1,16,#######,ESCAMOAL,MA,#######,#######,PK,DIMABA,,...,PREIS,ENVBX,19545870,19547746,$nag$00$npe$4327$spe$P62$epe$2026$org$5460$end$,DIMABA,0,,,
4,1,81,#######,ESCAMOAL,MA,#######,#######,PS,SIJENA,,...,PREIS,ENVBX,19550081,19550268,$nag$00$npe$4342$spe$P62$epe$2026$org$5460$end$,SIJENA,0,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18076,1,15005,#######,ESCAMOAL,MA,#######,#######,PF,CAXIOR,,...,PREIS,ENVBX,19864026,19864027,$nag$00$npe$13689$spe$P62$epe$2026$org$5460$end$,CAXIOR,0,,,
18077,1,16420,#######,ESCAMOAL,MA,#######,#######,0K,SAROJA,,...,PREIS,ENVBX,19887779,19887780,$nag$00$npe$3081$spe$PC2$epe$2026$org$5460$end$,SAROJA,0,,,
18078,1,16421,#######,ESCAMOAL,MA,#######,#######,0K,SAROJA,,...,PREIS,ENVBX,19887799,19887803,$nag$00$npe$3082$spe$PC2$epe$2026$org$5460$end$,SAROJA,0,,,
18079,1,16422,#######,ESCAMOAL,MA,#######,#######,PB,ESMEEDU,,...,PREIS,ENVBX,19885290,19885291,$nag$00$npe$3093$spe$PC2$epe$2026$org$5460$end$,ESMEEDU,0,,,


SELECT *
FROM RF_TAREAS_08
WHERE FECHA_GEN>=TO_DATE('05/05/2026','DD/MM/YYYY')

In [ ]:
#@title Embarque DataFrame
Embarque = gc.open(working_file).worksheet('Embarque')

rows = Embarque.get_all_values()+

Embarque_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
Embarque_df

,TAREA,EMPRESA,ORG_COMERCIAL,EJERCICIO_PEDIDO,SERIE_PEDIDO,NUMERO_PEDIDO,AGENCIA,ALMACEN,BOX_PREPARACION,ZONA_ALMACEN,...,DOMICILIO_ENVIO,NUMERO_PEDIDO_CLIENTE,ENTIDAD,MOSTRAR_PEDIDO_UNICO,MOVIMIENTO_EMBALADO,DOCUMENTO_EMBALADO,ID_AGRUPACION,SERIE_HP,BOX_EXPEDICION,TRABAJADOR
0,1,1,5460,2026,P62,4319,,5460D,,VR,...,1,149581,IFA127001,S,EMBIS,19554032,$nag$00$npe$4319$spe$P62$epe$2026$org$5460$end$,,EM1,IRCRDI
1,61,1,5460,2026,P62,4381,,5460D,VR2,VR,...,1,149715,IFA127006,S,EMBIS,19550037,$nag$00$npe$4381$spe$P62$epe$2026$org$5460$end$,,EM3,OLGOME
2,141,1,5460,2026,P62,4324,,5460D,VR1,VR,...,1,149686,49948,S,EMBIS,19550940,$nag$00$npe$4324$spe$P62$epe$2026$org$5460$end$,,EM1,AVLEHE
3,182,1,5460,2026,P62,4398,,5460D,VR5,VR,...,1,149629,IFA127005,S,EMBIS,19553053,$nag$00$npe$4398$spe$P62$epe$2026$org$5460$end$,,EM2,ARMORE
4,246,1,5460,2026,P62,4337,,5460D,VR4,VR,...,1,149605,1270007,S,EMBIS,19554414,$nag$00$npe$4337$spe$P62$epe$2026$org$5460$end$,,EM2,LIREGU
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9924,22388,1,5460,2026,PC2,3456,,5460D,VR7,VR,...,1,159811,49962,S,EMBIS,19930299,$nag$00$npe$3456$spe$PC2$epe$2026$org$5460$end$,,EM1,IRCRDI
9925,22602,1,5460,2026,PC2,3578,,5460D,VR7,VR,...,1,160003,49962,S,EMBIS,19931852,$nag$00$npe$3578$spe$PC2$epe$2026$org$5460$end$,,EM1,IRCRDI
9926,22719,1,5460,2026,P62,15220,,5460D,VR5,VR,...,1,N/A,49962,S,EMBIS,,$nag$00$npe$15220$spe$P62$epe$2026$org$5460$end$,,EM1,
9927,22765,1,5460,2026,PC2,3643,,5460D,VR6,VR,...,1,160142,49962,S,EMBIS,19934058,$nag$00$npe$3643$spe$PC2$epe$2026$org$5460$end$,,EM1,IRCRDI
